In [1]:
from utils.random_forest_utils.rf_preprocessing_utils import WindowAlgPreprocessor

rf_preprocessor_clamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/clamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_clamping, target_df_clamping = rf_preprocessor_clamping.read_data()
sensors_df_clamping = rf_preprocessor_clamping.feature_selection()
rf_preprocessor_clamping.normalize_angle()
rf_preprocessor_bending = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/bending_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_bending, target_df_bending = rf_preprocessor_bending.read_data()
sensors_df_bending = rf_preprocessor_bending.feature_selection()
rf_preprocessor_bending.normalize_angle()
rf_preprocessor_declamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/declamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_declamping, target_df_declamping = rf_preprocessor_declamping.read_data()
sensors_df_declamping = rf_preprocessor_declamping.feature_selection()
rf_preprocessor_declamping.normalize_angle()

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.900079,0.313091,0.999954,0.313091
1,2,0.022017,0.908296,0.425584,0.834481,0.425584
2,2,0.044033,0.905549,0.591178,0.582994,0.591178
3,2,0.066050,0.901145,0.751356,0.338801,0.751356
4,2,0.088067,0.909703,0.867774,0.167582,0.867774
...,...,...,...,...,...,...
14558,318,0.902686,0.928583,0.029142,0.997804,0.029142
14559,318,0.924703,0.922565,0.025609,1.000000,0.025609
14560,318,0.946720,0.921322,0.032466,0.992607,0.032466
14561,318,0.968736,0.925369,0.051137,0.974276,0.051137


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy

def resample_experiment_fast(group, n=46, metric='mean'):
    """
    Optimized resampling function using vectorized operations.
    Up to 10-100x faster than the original implementation.
    """
    # Sort by time
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign each row to a time bin
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    bin_indices = np.digitize(time_col, time_bins[:-1]) - 1
    bin_indices = np.clip(bin_indices, 0, n - 1)
    
    # Get experiment ID
    exp_id = group['Experiment_ID'].iloc[0]
    
    # Select numeric columns only
    cols_to_process = [col for col in group.columns 
                       if col not in ['Time_[s]', 'Experiment_ID']]
    
    results = []
    
    # Process each bin
    for bin_idx in range(n):
        mask = bin_indices == bin_idx
        if not mask.any():
            continue
            
        row_data = {'Experiment_ID': exp_id}
        
        for col in cols_to_process:
            values = group[col].values[mask]
            if len(values) == 0:
                continue
            
            # Compute metric using vectorized operations
            if metric == 'mean':
                row_data[f'{col}_mean'] = values.mean()
            elif metric == 'median':
                row_data[f'{col}_median'] = np.median(values)
            elif metric == 'min':
                row_data[f'{col}_min'] = values.min()
            elif metric == 'max':
                row_data[f'{col}_max'] = values.max()
            elif metric == 'range':
                row_data[f'{col}_range'] = values.ptp()
            elif metric == 'std':
                row_data[f'{col}_std'] = values.std()
            elif metric == 'var':
                row_data[f'{col}_var'] = values.var()
            elif metric == 'mad':
                row_data[f'{col}_mad'] = np.abs(values - values.mean()).mean()
            elif metric == 'rms':
                row_data[f'{col}_rms'] = np.sqrt((values ** 2).mean())
            elif metric == 'skew':
                row_data[f'{col}_skew'] = skew(values)
            elif metric == 'kurtosis':
                row_data[f'{col}_kurtosis'] = kurtosis(values)
            elif metric == 'energy':
                row_data[f'{col}_energy'] = (values ** 2).sum()
            elif metric == 'entropy':
                abs_vals = np.abs(values)
                probs = abs_vals / (abs_vals.sum() + 1e-12)
                row_data[f'{col}_entropy'] = entropy(probs + 1e-12)
            elif metric == 'cv':
                row_data[f'{col}_cv'] = values.std() / (values.mean() + 1e-12)
            elif metric == 'iqr':
                row_data[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            elif metric == 'p25':
                row_data[f'{col}_p25'] = np.percentile(values, 25)
            elif metric == 'p75':
                row_data[f'{col}_p75'] = np.percentile(values, 75)
            elif metric == 'trend_slope':
                if len(values) > 1:
                    row_data[f'{col}_trend_slope'] = np.polyfit(np.arange(len(values)), values, 1)[0]
                else:
                    row_data[f'{col}_trend_slope'] = 0
        
        results.append(row_data)
    
    return pd.DataFrame(results)


# Alternative: Ultra-fast version using pandas groupby (even faster for 'mean', 'std', 'min', 'max')
def resample_experiment_ultrafast(group, n=46, metric='mean'):
    """
    Ultra-optimized version using pandas groupby operations.
    Works best for basic metrics like mean, std, min, max, median.
    """
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign bins
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    group['_bin'] = np.digitize(time_col, time_bins[:-1]) - 1
    group['_bin'] = group['_bin'].clip(0, n - 1)
    
    # Select columns to aggregate
    cols_to_agg = [col for col in group.columns 
                   if col not in ['Time_[s]', 'Experiment_ID', '_bin']]
    
    # Map metric to pandas aggregation function
    agg_func_map = {
        'mean': 'mean',
        'median': 'median',
        'min': 'min',
        'max': 'max',
        'std': 'std',
        'var': 'var',
        'sum': 'sum'
    }
    
    if metric in agg_func_map:
        # Use fast pandas groupby
        result = group.groupby('_bin')[cols_to_agg].agg(agg_func_map[metric])
        result = result.add_suffix(f'_{metric}')
        result['Experiment_ID'] = group['Experiment_ID'].iloc[0]
        return result.reset_index(drop=True)
    else:
        # Fall back to custom implementation
        return resample_experiment_fast(group.drop('_bin', axis=1), n, metric)


# Usage - choose the best function for your needs:

# Option 2: Ultra-fast version (best for mean, std, min, max, median)
df_bending = sensors_df_bending.reset_index()
df_resampled_bending = (
    df_bending.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=450, metric='mean'))
    .reset_index(drop=True)
)

# Apply to all datasets
df_clamping = sensors_df_clamping.reset_index()
df_resampled_clamping = (
    df_clamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=450, metric='mean'))
    .reset_index(drop=True)
)

df_declamping = sensors_df_declamping.reset_index()
df_resampled_declamping = (
    df_declamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=450, metric='mean'))
    .reset_index(drop=True)
)

/tmp/ipykernel_242188/1413670700.py:135: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=10, metric='mean'))
/tmp/ipykernel_242188/1413670700.py:143: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=10, metric='mean'))
/tmp/ipykernel_242188/1413670700.py:150: FutureWarning: DataFrameGroupBy.apply operated on the groupin

In [3]:
def normalize_experiment(group, n=46):
    if len(group) > n:
        # Just take the first 46 rows
        return group.iloc[:n].copy()
    else:
        # Already 46 rows
        return group.copy()

# Apply to each experiment
df_normalized_clamping = target_df_clamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_clamping = df_normalized_clamping.reset_index(drop=True)
df_normalized_bending = target_df_bending.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_bending = df_normalized_bending.reset_index(drop=True)
df_normalized_declamping = target_df_declamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
df_normalized_declamping = df_normalized_declamping.reset_index(drop=True)

/tmp/ipykernel_242188/3001301495.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized_clamping = target_df_clamping.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
/tmp/ipykernel_242188/3001301495.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized_bending = target_df_bending.groupby('Experiment_ID', group_keys=False).apply(normalize_experiment, n=46)
/tmp/ipy

In [4]:
X_clamping = rf_preprocessor_clamping.group_and_pad(df_resampled_clamping, group_col="Experiment_ID")[55:,:, :]
Y_clamping = rf_preprocessor_clamping.group_and_pad(df_normalized_clamping, group_col="Experiment_ID")[55:,:-1:,1:]

X_bending = rf_preprocessor_bending.group_and_pad(df_resampled_bending, group_col="Experiment_ID")[55:,:, :]
Y_bending = rf_preprocessor_bending.group_and_pad(df_normalized_bending, group_col="Experiment_ID")[55:,:-1:,1:]

X_declamping = rf_preprocessor_declamping.group_and_pad(df_resampled_declamping, group_col="Experiment_ID")[:,:, :]
Y_declamping = rf_preprocessor_declamping.group_and_pad(df_normalized_declamping, group_col="Experiment_ID")[55:,:-1:,1:]

In [12]:
import ipywidgets as widgets
from ipywidgets import interact
import os
import csv
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from datetime import datetime

def event_plot(X, Y, event_name, angle_index, importance_method='gini'):
    """
    importance_method options:
    - 'gini': Gini impurity (default RandomForest feature_importances_)
    - 'permutation': Permutation importance (more reliable but slower)
    - 'drop_column': Drop-column importance (most accurate but slowest)
    - 'shap': SHAP values (requires shap library)
    """
    # --- Data shapes ---
    n_experiments, n_timesteps, n_features = X.shape
    Y_angle = Y[:, angle_index:angle_index+1, :]
    _, _, n_targets = Y_angle.shape
    
    # --- Train main RandomForest ---
    rf_params = {"n_estimators": 500, "random_state": 42, "n_jobs": -1}
    rf = RandomForestRegressor(**rf_params)
    X_flat = X.reshape(n_experiments, n_timesteps * n_features)
    rf.fit(X_flat, Y_angle[:, 0, :])
    
    # --- Compute feature importance based on method ---
    print(f"Computing feature importance using: {importance_method}")
    
    if importance_method == 'gini':
        feature_importances = rf.feature_importances_
        
    elif importance_method == 'permutation':
        # Permutation importance - shuffles features and measures performance drop
        perm_importance = permutation_importance(
            rf, X_flat, Y_angle[:, 0, :], 
            n_repeats=10, 
            random_state=42, 
            n_jobs=-1
        )
        feature_importances = perm_importance.importances_mean
        
    elif importance_method == 'drop_column':
        # Drop-column importance - trains without each feature
        # Using smaller model for speed
        rf_small = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=10)
        rf_small.fit(X_flat, Y_angle[:, 0, :])
        baseline_score = rf_small.score(X_flat, Y_angle[:, 0, :])
        feature_importances = np.zeros(n_timesteps * n_features)
        
        print(f"Computing drop-column importance (this may take a while for {n_timesteps * n_features} features)...")
        for i in range(n_timesteps * n_features):
            if i % 50 == 0:
                print(f"Progress: {i}/{n_timesteps * n_features}")
            X_dropped = np.delete(X_flat, i, axis=1)
            rf_temp = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=10)
            rf_temp.fit(X_dropped, Y_angle[:, 0, :])
            dropped_score = rf_temp.score(X_dropped, Y_angle[:, 0, :])
            feature_importances[i] = max(0, baseline_score - dropped_score)
        print("Drop-column computation complete!")
            
    elif importance_method == 'shap':
        try:
            import shap
            explainer = shap.TreeExplainer(rf)
            shap_values = explainer.shap_values(X_flat)
            # Average absolute SHAP values across samples and targets
            if isinstance(shap_values, list):
                feature_importances = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
            else:
                feature_importances = np.abs(shap_values).mean(axis=0)
        except ImportError:
            print("SHAP library not installed. Install with: pip install shap")
            print("Falling back to Gini importance...")
            feature_importances = rf.feature_importances_
            
    else:
        raise ValueError(f"Unknown importance method: {importance_method}")
    
    # Normalize importances to sum to 1
    feature_importances = feature_importances / feature_importances.sum()
    
    # --- Timestep-level importance ---
    timestep_importances = feature_importances.reshape(n_timesteps, n_features).sum(axis=1)
    top4_idx = np.argsort(timestep_importances)[-4:][::-1]
    top4_percent = (timestep_importances[top4_idx] / timestep_importances.sum()) * 100
    best_timestep = top4_idx[0]
    
    print(f"Top 4 timesteps: {top4_idx} with percentages: {top4_percent}")
    
    # --- Target-level importance ---
    target_importances = []
    for target_idx in range(n_targets):
        rf_target = RandomForestRegressor(**rf_params)
        rf_target.fit(X_flat, Y_angle[:, 0, target_idx])
        
        if importance_method == 'gini':
            target_imp = rf_target.feature_importances_
        elif importance_method == 'permutation':
            perm = permutation_importance(
                rf_target, X_flat, Y_angle[:, 0, target_idx],
                n_repeats=10, random_state=42, n_jobs=-1
            )
            target_imp = perm.importances_mean
        elif importance_method == 'drop_column':
            baseline = rf_target.score(X_flat, Y_angle[:, 0, target_idx])
            target_imp = np.zeros(n_timesteps * n_features)
            rf_small = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=10)
            for i in range(n_timesteps * n_features):
                X_dropped = np.delete(X_flat, i, axis=1)
                rf_small.fit(X_dropped, Y_angle[:, 0, target_idx])
                target_imp[i] = max(0, baseline - rf_small.score(X_dropped, Y_angle[:, 0, target_idx]))
        elif importance_method == 'shap':
            try:
                import shap
                explainer = shap.TreeExplainer(rf_target)
                shap_vals = explainer.shap_values(X_flat)
                target_imp = np.abs(shap_vals).mean(axis=0)
            except:
                target_imp = rf_target.feature_importances_
        
        target_importances.append(target_imp.reshape(n_timesteps, n_features).sum(axis=1))
    
    target_importances = np.array(target_importances)
    
    # --- Train secondary RF on best timestep ---
    X_best = X[:, best_timestep, :]
    rf_final = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_final.fit(X_best, Y_angle[:, 0, :])
    Y_pred = rf_final.predict(X_best)
    
    # --- Metrics ---
    mae_list = [mean_absolute_error(Y_angle[:, 0, i], Y_pred[:, i]) for i in range(n_targets)]
    r2_list = [r2_score(Y_angle[:, 0, i], Y_pred[:, i]) for i in range(n_targets)]
    print(f"Angle {angle_index} - MAE per target: {mae_list}")
    print(f"Angle {angle_index} - R² per target: {r2_list}")
    
    # --- Plot timestep-level importance (line plot) ---
    plt.figure(figsize=(14, 6))
    colors = plt.cm.tab20(np.linspace(0, 1, n_features))
    
    # Assuming df_resampled_declamping is available in scope
    try:
        feature_names = list(df_resampled_declamping.columns)[:-1]
    except:
        feature_names = [f"Feature {i}" for i in range(n_features)]
    
    for f in range(n_features):
        plt.plot(range(n_timesteps), X[:, :, f].mean(axis=0), 
                color=colors[f], alpha=0.5, label=feature_names[f])
    
    for t, p in zip(top4_idx, top4_percent):
        plt.axvline(t, color='red', linestyle='--', linewidth=2, alpha=0.6)
        plt.text(t, plt.ylim()[1]*0.95, f"{p:.1f}%", 
                color='red', rotation=90, verticalalignment='top')
    
    plt.xlabel("Timestep")
    plt.ylabel("Sensor value (average over experiments)")
    plt.title(f"{event_name.capitalize()} - Top 4 predictive timesteps (Angle {angle_index}) - Method: {importance_method}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
    # --- Plot target-wise importance as subplots ---
    fig, axes = plt.subplots(n_targets, 1, figsize=(14, 3*n_targets), sharex=True)
    if n_targets == 1:
        axes = [axes]
    
    x = np.arange(n_timesteps)
    
    try:
        target_names = list(df_normalized_declamping.columns)[2:]
    except:
        target_names = [f"Target {i+1}" for i in range(n_targets)]
    
    for i, ax in enumerate(axes):
        ax.bar(x, target_importances[i], color='skyblue')
        ax.set_ylabel(target_names[i])
        ax.set_title(f"Target {i+1} importance (Angle {angle_index}) - Method: {importance_method}")
    
    axes[-1].set_xlabel("Timestep")
    plt.tight_layout()
    plt.show()
    
    return timestep_importances, target_importances


# --- Interactive widget for method selection ---
def create_interactive_plot(X, Y, event_name, angle_index):
    """Create an interactive widget to select importance method"""
    interact(
        lambda method: event_plot(X, Y, event_name, angle_index, method),
        method=widgets.Dropdown(
            options=['gini', 'permutation', 'drop_column', 'shap'],
            value='gini',
            description='Method:',
            style={'description_width': 'initial'}
        )
    )

# --- Complete interactive interface with all options ---
datasets = {
    "clamping": (X_clamping, Y_clamping),
    "bending": (X_bending, Y_bending),
    "declamping": (X_declamping, Y_declamping)
}

def interactive_event_plot(dataset_name, angle_index, importance_method):
    X, Y = datasets[dataset_name]
    event_plot(X, Y, dataset_name, angle_index, importance_method)

# Create widgets
dataset_widget = widgets.Dropdown(
    options=list(datasets.keys()),
    value="clamping",
    description="Dataset:"
)

angle_widget = widgets.IntSlider(
    min=0,
    max=Y_clamping.shape[1]-1,
    step=1,
    value=0,
    description="Angle:"
)

method_widget = widgets.Dropdown(
    options=['gini', 'permutation', 'drop_column', 'shap'],
    value='gini',  # Changed default to permutation (faster and more reliable)
    description='Method:',
    style={'description_width': 'initial'}
)

# Update max value of angle slider dynamically based on dataset
def update_angle_range(*args):
    X, Y = datasets[dataset_widget.value]
    angle_widget.max = Y.shape[1]-1

dataset_widget.observe(update_angle_range, 'value')

# Create interactive interface
interact(
    interactive_event_plot, 
    dataset_name=dataset_widget, 
    angle_index=angle_widget,
    importance_method=method_widget
)

interactive(children=(Dropdown(description='Dataset:', options=('clamping', 'bending', 'declamping'), value='c…

<function __main__.interactive_event_plot(dataset_name, angle_index, importance_method)>